In [ ]:
#!pip install scikit-learn
#!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 6.9 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler

In [2]:
df_tracks = pd.read_csv("dataset/spotify_clean_data.csv")
df_interactions = pd.read_csv("dataset/interactions.csv")

In [3]:
audio_cols = ['track_popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'valence', 'tempo', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'duration_ms']
for col in audio_cols:
    df_tracks[col] = pd.to_numeric(df_tracks[col], errors='coerce')

In [4]:
df_merged = pd.merge(df_interactions, df_tracks, on='track_id', how='left')

In [5]:
df_merged['weight'] = df_merged['duration_ms_dataset'] / df_merged['duration_ms']
df_merged['weight'] = df_merged['weight'].clip(0.0, 1.0) #min 0 and max 1.0

In [6]:
print(f"Total Interactions rows: {len(df_merged)}")

Total Interactions rows: 2519817


In [ ]:
#MAPPING INFO TO FEED THE NEURAL NETWORK
unique_track_ids = df_merged['track_id'].unique()
track_map = {id: i+1 for i, id in enumerate(unique_track_ids)} 

unique_genres = df_merged['playlist_genre'].unique()
genre_map = {g: i+1 for i, g in enumerate(unique_genres)}

df_merged['track_idx'] = df_merged['track_id'].map(track_map)
df_merged['genre_idx'] = df_merged['playlist_genre'].map(genre_map)

#i'm using the username the same as the user_id, but just for testing, in real case scenario the username would be a string.
unique_users = df_merged['username'].unique()
user_map = {u: i for i, u in enumerate(unique_users)} 
df_merged['user_idx'] = df_merged['username'].map(user_map)  

In [ ]:
#normalizing data 
scaler = MinMaxScaler()
df_merged[['tempo', 'loudness', 'key', 'duration_ms']] = scaler.fit_transform(df_merged[['tempo', 'loudness', 'key', 'duration_ms']]) 

#selecting Audio Features
feature_cols = [
    'danceability',      
    'energy',            
    'valence',           
    'loudness',          #normalized
    'tempo',             #normalized
    'acousticness',      
    'instrumentalness',  
    'liveness',          
    'speechiness',       
    'mode',
    'key',               #normalized
    'duration_ms'        #normalized      
]
df_merged['audio_vector'] = df_merged[feature_cols].values.tolist()
print(f"Vector Size: {len(feature_cols)}")

Vector Size: 12


In [ ]:

# Save Mappings 
with open("mappings.pkl", "wb") as f:
    pickle.dump({"track_map": track_map, "genre_map": genre_map, "user_map": user_map}, f) #to decode
print("Mappings saved to mappings.pkl")

# Prepare "Sequence" Data
# We need to group by User to get their history
# Result: user_id | [song_1, song_2, ... song_99] | target_song_100
print("Grouping User History...")
df_merged = df_merged.sort_values(by=['user_idx'])
user_groups = df_merged.groupby('user_idx')

HISTORY_SIZE = 10
sequences = []
for user_id, group in user_groups:
    tracks = group['track_idx'].values
    genres = group['genre_idx'].values
    ratios = group['weight'].values
    audios = np.stack(group['audio_vector'].values)
    
    for i in range(HISTORY_SIZE, len(tracks)):
        hist_tracks = tracks[i-HISTORY_SIZE:i]
        hist_genres = genres[i-HISTORY_SIZE:i]
        hist_ratios = ratios[i-HISTORY_SIZE:i]
        hist_audio  = audios[i-HISTORY_SIZE:i]
        
        target_track = tracks[i]
        target_genre = genres[i]
        target_audio = audios[i]
        
        sequences.append((hist_tracks, hist_genres, hist_ratios, hist_audio, 
                          target_track, target_genre, target_audio))

print(f"Created {len(sequences)} training sequences.")

Mappings saved to mappings.pkl
Grouping User History...
Created 2419817 training sequences.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerModel(nn.Module):
    def __init__(self, 
                 num_tracks: int, 
                 num_genres: int, 
                 audio_feature_dim: int = 12, 
                 embedding_dim: int = 64):
        super().__init__()
        
        # --- Layers ---
        self.track_embedding = nn.Embedding(num_tracks, embedding_dim)
        self.genre_embedding = nn.Embedding(num_genres, embedding_dim)
        
        self.audio_mlp = nn.Sequential(
            nn.Linear(audio_feature_dim, 32),
            nn.ReLU(),
            nn.Linear(32, embedding_dim) 
        )
        
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim * 3, embedding_dim * 2),
            nn.ReLU(),
            nn.Linear(embedding_dim * 2, embedding_dim)
        )
        
        # This helps the model sharpen its predictions. 
        # Start at 0.07 (standard for contrastive learning)
        self.temperature = nn.Parameter(torch.tensor(0.07))

    def get_item_representation(self, track_ids, genre_ids, audio_features):
        track_emb = self.track_embedding(track_ids) 
        genre_emb = self.genre_embedding(genre_ids) 
        audio_emb = self.audio_mlp(audio_features)  
        
        combined = torch.cat([track_emb, genre_emb, audio_emb], dim=-1)
        vector = self.projection(combined)
        
        # L2 normalization
        # Force the vector to have length 1. 
        # Prevents the dot product from exploding.
        return F.normalize(vector, p=2, dim=1)

    def forward(self, h_track_ids, h_genre_ids, h_audio_features, h_weights,
                t_track_id, t_genre_id, t_audio_feature):
        
        # User Vector 
        batch_size, seq_len = h_track_ids.shape
        flat_h_track = h_track_ids.view(-1)
        flat_h_genre = h_genre_ids.view(-1)
        flat_h_audio = h_audio_features.view(-1, h_audio_features.shape[-1])
        
        flat_h_vectors = self.get_item_representation(flat_h_track, flat_h_genre, flat_h_audio)
        h_vectors = flat_h_vectors.view(batch_size, seq_len, -1)
        
        weights_expanded = h_weights.unsqueeze(-1)
        weighted_history = h_vectors * weights_expanded
        user_vector_raw = weighted_history.sum(dim=1) / (weights_expanded.sum(dim=1) + 1e-8)
        
        # Normalize User Vector 
        user_vector = F.normalize(user_vector_raw, p=2, dim=1)
        
        # Target Item Vector 
        target_vector = self.get_item_representation(t_track_id, t_genre_id, t_audio_feature)
        
        # SCALED DOT PRODUCT 
        # Cosine Similarity / Temperature
        # If vectors are normalized, Dot Product IS Cosine Similarity.
        # We divide by temperature to scale the logits up for CrossEntropy.
        return user_vector, target_vector, self.temperature

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

#config - hyperParams
BATCH_SIZE = 256       
EMBEDDING_DIM = 64     
LEARNING_RATE = 1e-3
EPOCHS = 5             
AUDIO_DIM = 12          # len of audio_cols

# check device
if torch.backends.mps.is_available():
    device = torch.device("mps") # I trained using Macbook AIR M4
    print("GPU")
else:
    device = torch.device("cpu")
    print("CPU")

class SpotifyDataset(Dataset):
    def __init__(self, sequences):
        self.data = sequences

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Unpack the sequence [hist_t, hist_g, hist_w, hist_a, tgt_t, tgt_g, tgt_a]
        row = self.data[idx]
        
        return (
            torch.tensor(row[0], dtype=torch.long),      # History Tracks
            torch.tensor(row[1], dtype=torch.long),      # History Genres
            torch.tensor(row[3], dtype=torch.float32),   # History Audio 
            torch.tensor(row[2], dtype=torch.float32),   # History Weights
            torch.tensor(row[4], dtype=torch.long),      # Target Track
            torch.tensor(row[5], dtype=torch.long),      # Target Genre
            torch.tensor(row[6], dtype=torch.float32)    # Target Audio
        )


train_ds = SpotifyDataset(sequences)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

num_tracks = len(track_map) + 1  
num_genres = len(genre_map) + 1

model = TwoTowerModel(num_tracks, num_genres, audio_feature_dim=AUDIO_DIM, embedding_dim=EMBEDDING_DIM)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

print(f"Starting training on {len(train_ds)} examples...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (h_t, h_g, h_a, h_w, t_t, t_g, t_a) in enumerate(train_loader):
        h_t, h_g, h_a, h_w = h_t.to(device), h_g.to(device), h_a.to(device), h_w.to(device)
        t_t, t_g, t_a = t_t.to(device), t_g.to(device), t_a.to(device)
        
        optimizer.zero_grad()
        
        user_vec, item_vec, temperature = model(h_t, h_g, h_a, h_w, t_t, t_g, t_a)

        # Calculate Scores
        logits = torch.matmul(user_vec, item_vec.T)

        #Divide by temperature (be sure to keep in a good range -1 to 1)
        logits = logits / temperature

        # Calculate Loss
        labels = torch.arange(BATCH_SIZE).to(device)
        loss = loss_fn(logits, labels)
        
        # Loss & Backward
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), f"spotify_two_tower_history{HISTORY_SIZE}.pth")
print("Model Saved!")

GPU
Starting training on 2419817 examples...
Epoch 1 | Batch 0 | Loss: 6.0959
Epoch 1 | Batch 100 | Loss: 5.4831
Epoch 1 | Batch 200 | Loss: 5.4621
Epoch 1 | Batch 300 | Loss: 5.4105
Epoch 1 | Batch 400 | Loss: 5.2855
Epoch 1 | Batch 500 | Loss: 5.3452
Epoch 1 | Batch 600 | Loss: 5.1993
Epoch 1 | Batch 700 | Loss: 5.1284
Epoch 1 | Batch 800 | Loss: 5.2480
Epoch 1 | Batch 900 | Loss: 5.1314
Epoch 1 | Batch 1000 | Loss: 4.9375
Epoch 1 | Batch 1100 | Loss: 5.0877
Epoch 1 | Batch 1200 | Loss: 5.0331
Epoch 1 | Batch 1300 | Loss: 4.9319
Epoch 1 | Batch 1400 | Loss: 4.9965
Epoch 1 | Batch 1500 | Loss: 4.9229
Epoch 1 | Batch 1600 | Loss: 4.9071
Epoch 1 | Batch 1700 | Loss: 4.9896
Epoch 1 | Batch 1800 | Loss: 4.9579
Epoch 1 | Batch 1900 | Loss: 4.8168
Epoch 1 | Batch 2000 | Loss: 4.8276
Epoch 1 | Batch 2100 | Loss: 4.9069
Epoch 1 | Batch 2200 | Loss: 4.8617
Epoch 1 | Batch 2300 | Loss: 4.7447
Epoch 1 | Batch 2400 | Loss: 4.8641
Epoch 1 | Batch 2500 | Loss: 4.7213
Epoch 1 | Batch 2600 | Loss: 4.

In [ ]:
import faiss

# Preparing the Catalog
# We need every unique song, sorted by track_idx so we can match FAISS IDs later
# We filter out track_idx=0 because that was just padding
print("Preparing Song Catalog...")
df_catalog = df_merged.drop_duplicates(subset=['track_idx']).sort_values('track_idx') #to create 1 vector per song
df_catalog = df_catalog[df_catalog['track_idx'] > 0] 

# Generate Vectors for all songs
# Doing this in batches just to be safe with RAM
batch_size = 1000
num_songs = len(df_catalog)
all_song_vectors = []

model.eval()
with torch.no_grad():
    for start_idx in range(0, num_songs, batch_size):
        end_idx = min(start_idx + batch_size, num_songs)
        batch = df_catalog.iloc[start_idx:end_idx]
        
        # Prepare inputs
        t_t = torch.tensor(batch['track_idx'].values, dtype=torch.long).to(device)
        t_g = torch.tensor(batch['genre_idx'].values, dtype=torch.long).to(device)
        
        # Stack audio features
        audio_list = np.stack(batch['audio_vector'].values)
        t_a = torch.tensor(audio_list, dtype=torch.float32).to(device)
        
        # Get Vector
        vectors = model.get_item_representation(t_t, t_g, t_a)
        
        # Move to CPU and Numpy
        all_song_vectors.append(vectors.cpu().numpy())

# Concatenate into one giant matrix
# Shape: [Num_Songs, 64]
song_matrix = np.concatenate(all_song_vectors, axis=0)
print(f"Generated vectors for {song_matrix.shape[0]} songs.")

# Build FAISS Index 
# Since vectors are normalized, Inner Product == Cosine Similarity.
index = faiss.IndexFlatIP(64) 
index.add(song_matrix)

# We need to save the Index AND the DataFrame (to map ID -> Name)
faiss.write_index(index, f"spotify_songs.index_history{HISTORY_SIZE}")
df_catalog.to_pickle(f"song_catalog_history{HISTORY_SIZE}.pkl")

print("FAISS Index build and saved!")

Preparing Song Catalog...
Generated vectors for 15686 songs.
FAISS Index build and saved!


In [ ]:
def load_production_system():
    """Loads the model, index, and catalog once."""
    print("Loading System...")

    index = faiss.read_index(f"spotify_songs.index_history{HISTORY_SIZE}")
    
    # Load Catalog (The mapping from FAISS ID -> Real Song Name)
    catalog = pd.read_pickle(f"song_catalog_history{HISTORY_SIZE}.pkl")
    catalog = catalog.reset_index(drop=True)
    
    # 3. Load Model
    # (Assuming 'model' is already loaded in your session, but in prod you'd load state_dict)
    
    return index, catalog

# Initialize (Run this once on startup)
faiss_index, song_catalog = load_production_system()

def recommend_with_faiss(user_idx, df_history, k=5):
    """
    1. Generates User Vector
    2. Asks FAISS for top K neighbors
    3. Returns Song Names
    """
    # GENERATE USER VECTOR
    user_rows = df_history[df_history['user_idx'] == user_idx].tail(HISTORY_SIZE)
    
    # Prepare Inputs
    h_t = torch.tensor([user_rows['track_idx'].values], dtype=torch.long).to(device)
    h_g = torch.tensor([user_rows['genre_idx'].values], dtype=torch.long).to(device)
    h_w = torch.tensor([user_rows['weight'].values], dtype=torch.float32).to(device)
    audio_list = np.stack(user_rows['audio_vector'].values)
    h_a = torch.tensor([audio_list], dtype=torch.float32).to(device)
    
    model.eval()
    with torch.no_grad():
        # Dummy targets to run the model
        dummy_t = h_t[:, 0]
        dummy_g = h_g[:, 0]
        dummy_a = h_a[:, 0, :]
        
        # Get User Vector
        user_vec, _, _ = model(h_t, h_g, h_a, h_w, dummy_t, dummy_g, dummy_a)
        # Convert to Numpy for FAISS
        user_vector_np = user_vec.cpu().numpy()

    # FAISS SEARCH 
    # D = Distances (Scores), I = Indices (Row Numbers in Catalog)
    # search(queries, k)
    D, I = faiss_index.search(user_vector_np, k)
    
    # FORMAT RESULTS 
    print(f"\nRecommendations for User {user_idx}:")
    
    # I[0] is the list of indices for the first query
    recommended_indices = I[0]
    scores = D[0]
    
    for rank, (idx, score) in enumerate(zip(recommended_indices, scores)):
        # Map FAISS ID back to Song Data
        song_row = song_catalog.iloc[idx]
        
        print(f"#{rank+1} [Score {score:.3f}] {song_row['track_name']} - {song_row['track_artist']} - {song_row['playlist_genre'].capitalize()}")

recommend_with_faiss(user_idx=2, df_history=df_merged, k=10)

Loading System...


/var/folders/r7/my_v9gd52x55_wytcr8lhv1c0000gn/T/ipykernel_29036/1685221122.py:29: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  h_t = torch.tensor([user_rows['track_idx'].values], dtype=torch.long).to(device)



Recommendations for User 2:
#1 [Score 0.845] How's It Going to Be - Third Eye Blind - Rock
#2 [Score 0.812] No Rain - Blind Melon - Rock
#3 [Score 0.793] Santa Monica - Everclear - Rock
#4 [Score 0.784] Glycerine - Remastered - Bush - Rock
#5 [Score 0.782] All For You - Night Riots - Pop
#6 [Score 0.770] Champagne Supernova - Remastered - Oasis - Rock
#7 [Score 0.769] One Week - Barenaked Ladies - Pop
#8 [Score 0.764] Lovefool - Radio Edit - The Cardigans - Pop
#9 [Score 0.759] Interstate Love Song - Stone Temple Pilots - Rock
#10 [Score 0.750] Two Princes - Spin Doctors - Rock


In [39]:
user_rows = df_merged[df_merged["username"] == 2]
genre_counts = user_rows["playlist_genre"].value_counts()
print(genre_counts)

playlist_genre
pop      54
rap      34
latin    28
r&b      26
edm      16
rock     12
Name: count, dtype: int64


In [ ]:
last_songs = df_merged[df_merged['user_idx'] == 2].tail(HISTORY_SIZE)

print(last_songs[['track_name', 'playlist_genre', 'weight']])

                                   track_name playlist_genre  weight
462                      How's It Going to Be           rock     1.0
465                                     Slide            r&b     1.0
460                             Fade Into You            pop     1.0
459                                   Kiss Me          latin     1.0
461                              High And Dry            pop     1.0
458                     Lovefool - Radio Edit            pop     1.0
457                    Glycerine - Remastered           rock     1.0
456                              Santa Monica           rock     1.0
455  Absolutely (Story of a Girl) - Radio Mix           rock     1.0
454                                  One Week            pop     1.0
